# Iterative Quantum Amplitude Estimation (IQAE) for M-GAN memory search

This notebook estimates **how many keys in a memory database are near an input embedding** using
Iterative Quantum Amplitude Estimation (Grinko et al., 2019). It is meant to be called from an M-GAN
discriminator: the returned neighbour count (with a confidence interval) is the "real vs fake" signal.

**How it is encoded**

For a database of $M$ keys $k_i$ and a query $q$, a matching rule gives a flag $f_i \in \{0,1\}$ per key.
The state-prep operator $A$ produces

$$A|0\rangle = \sum_{i=0}^{M-1} \tfrac{1}{\sqrt M}\,|i\rangle\,|\text{work}=0\rangle\,|f_i\rangle_{\text{flag}},$$

so the probability of the flag being $|1\rangle$ is $a = \tfrac1M\sum_i f_i$ and the neighbour count is $a\cdot M$.
IQAE estimates $a$ with Grover powers $Q^k A$, where $Q = A S_0 A^\dagger S_\chi$.

**Backends**

| backend | how $f_i$ is produced | use |
|---|---|---|
| `"qram"` (default) | **coherently in the circuit**: select-style QRAM loads each key's LSH code, a reversible popcount computes the Hamming distance to the query, and a comparator sets the flag (Section 3) | the faithful quantum search |
| `"stateprep"` | classically computed flags folded into one state preparation (Section 2) | baseline that works for any matching rule |
| `"analytic"` | exact ideal-device statistics, $P(1\mid k) = \sin^2((2k+1)\theta)$ | fast training loops |

The matching rule for `"qram"` is **LSH + Hamming radius**: embeddings are hashed to $L$-bit SimHash codes (signs of
random projections, where the Hamming distance divided by $L$ approximates angle$/\pi$), and key $i$ is a neighbour iff
$\mathrm{Hamming}(b_i, q) \le r$.

**Caveat:** this is still a classical simulation of the quantum circuit, so there is no wall-clock speedup. With the
`"qram"` backend, though, the neighbour test is done by the circuit rather than handed to it.

**Integration point:** `QAEMemoryCounter` (a `torch.nn.Module`) in the hook section below. Call
`counter(query, keys)`. The forward value is the QAE estimate. Gradients flow through a smooth surrogate
(sigmoid-relaxed neighbour count), which acts as a straight-through estimator.

In [1]:
import math
import time
import warnings
from dataclasses import dataclass, field
from typing import Callable, List, NamedTuple, Optional

import numpy as np
import jax.numpy as jnp
import pennylane as qp
import catalyst
import torch

## 1. Matching rule (swappable)

Any object with `hard(query, keys) -> (B, M) flags in [0,1]` and `soft(query, keys) -> (B, M)` (differentiable)
works with the `stateprep` and `analytic` backends. The `qram` backend also needs `codes(x) -> (.., L)` bits and
`radius_bits` (`LSHHamming`), because it evaluates the rule coherently from the stored codes.

In [2]:
class RadiusNeighbors:
    """Mark keys within distance `radius` of the query.

    metric: "l2" (Euclidean) or "cosine" (cosine distance = 1 - cos_sim, in [0, 2]).
    temperature: sharpness of the sigmoid surrogate used for gradients (smaller = closer to the hard rule).
    """

    def __init__(self, radius: float, metric: str = "cosine", temperature: Optional[float] = None):
        if metric not in ("l2", "cosine"):
            raise ValueError("metric must be 'l2' or 'cosine'")
        self.radius = float(radius)
        self.metric = metric
        self.temperature = float(temperature) if temperature is not None else 0.1 * self.radius

    def distance(self, query: torch.Tensor, keys: torch.Tensor) -> torch.Tensor:
        """query (B, d), keys (M, d) -> (B, M)."""
        if self.metric == "l2":
            return torch.cdist(query, keys)
        qn = torch.nn.functional.normalize(query, dim=-1)
        kn = torch.nn.functional.normalize(keys, dim=-1)
        return 1.0 - qn @ kn.T

    def hard(self, query: torch.Tensor, keys: torch.Tensor) -> torch.Tensor:
        return (self.distance(query, keys) <= self.radius).to(query.dtype)

    def soft(self, query: torch.Tensor, keys: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid((self.radius - self.distance(query, keys)) / self.temperature)


class LSHHamming:
    """SimHash codes + Hamming radius (the rule evaluated coherently by the QRAM backend).

    codes(x) = [x . p_j > 0] for L random Gaussian hyperplanes p_j. Hamming(b_q, b_k) / L ~ angle(q, k) / pi.
    Neighbour iff Hamming <= radius_bits.

    Surrogate for gradients: each bit mismatch is relaxed to sigmoid(-(q.p_j)(k.p_j) / proj_temperature), and the
    soft Hamming distance h goes through sigmoid((radius_bits + 0.5 - h) / temperature).
    """

    def __init__(self, dim: int, n_bits: int = 16, radius_bits: int = 3, temperature: float = 0.5,
                 proj_temperature: float = 0.05, seed: int = 0):
        g = torch.Generator().manual_seed(seed)
        self.planes = torch.randn(dim, n_bits, generator=g)
        self.n_bits = n_bits
        self.radius_bits = int(radius_bits)
        self.temperature = temperature
        self.proj_temperature = proj_temperature

    def _proj(self, x: torch.Tensor) -> torch.Tensor:
        return torch.nn.functional.normalize(x, dim=-1) @ self.planes.to(x)

    def codes(self, x: torch.Tensor) -> torch.Tensor:
        return (self._proj(x) > 0).to(torch.uint8)

    def hamming(self, query: torch.Tensor, keys: torch.Tensor) -> torch.Tensor:
        return (self.codes(query)[:, None, :] != self.codes(keys)[None, :, :]).sum(-1)

    def hard(self, query: torch.Tensor, keys: torch.Tensor) -> torch.Tensor:
        return (self.hamming(query, keys) <= self.radius_bits).to(query.dtype)

    def soft(self, query: torch.Tensor, keys: torch.Tensor) -> torch.Tensor:
        pq, pk = self._proj(query)[:, None, :], self._proj(keys)[None, :, :]
        h = torch.sigmoid(-(pq * pk) / self.proj_temperature).sum(-1)
        return torch.sigmoid((self.radius_bits + 0.5 - h) / self.temperature)

## 2. Baseline: classical-flag state preparation (`backend="stateprep"`)

The flags are computed classically and folded into one amplitude vector. This works for any matching rule
and serves as the reference for the QRAM circuit.

Wire layout: wires `0..n-1` hold the key index (wire 0 = MSB), and the last wire is the "good" ancilla.
The amplitude vector is a runtime argument, so Catalyst compiles **once per (qubit count, shots)** and
reuses the compiled circuit for every query and every database state.

In [3]:
def num_index_qubits(M: int) -> int:
    return max(1, math.ceil(math.log2(M)))


def build_state(flags: np.ndarray) -> np.ndarray:
    """flags (M,) in [0,1] -> normalized amplitude vector over n_index + 1 qubits.

    Index i (uniform weight 1/M) is entangled with ancilla amplitude sqrt(f_i) on |1>.
    Padding indices (i >= M) get zero amplitude.
    """
    flags = np.clip(np.asarray(flags, dtype=np.float64), 0.0, 1.0)
    M = flags.shape[0]
    n = num_index_qubits(M)
    state = np.zeros(2 ** (n + 1), dtype=np.float64)
    state[0 : 2 * M : 2] = np.sqrt((1.0 - flags) / M)  # |i>|0>
    state[1 : 2 * M : 2] = np.sqrt(flags / M)          # |i>|1>
    return state


_CIRCUIT_CACHE = {}


def get_grover_circuit(n_wires: int, shots: int, use_catalyst: bool = True):
    """Returns circuit(state, k) -> probs of the ancilla after Q^k A |0>."""
    key = (n_wires, shots, use_catalyst)
    if key in _CIRCUIT_CACHE:
        return _CIRCUIT_CACHE[key]

    wires = list(range(n_wires))
    anc = n_wires - 1
    dev = qp.device("lightning.qubit", wires=n_wires)

    def A(state):
        # Mottonen decomposes into RY/CNOT gates, so A (and A^dagger) can be applied mid-circuit,
        # unlike a raw StatePrep, which devices only accept at the start of a circuit.
        qp.MottonenStatePreparation(state, wires=wires)

    def Q(state):
        qp.PauliZ(wires=anc)            # S_chi: phase -1 on good states (ancilla = |1>)
        qp.adjoint(A)(state)            # A^dagger
        qp.FlipSign(0, wires=wires)     # S_0 (up to global phase): phase -1 on |0...0>
        A(state)                        # A

    if use_catalyst:
        @qp.qnode(dev, shots=shots)
        def circuit(state, k):
            A(state)

            @catalyst.for_loop(0, k, 1)
            def apply_Q(i):
                Q(state)

            apply_Q()
            return qp.probs(wires=[anc])

        circuit = catalyst.qjit(circuit)
    else:
        @qp.qnode(dev, shots=shots)
        def circuit(state, k):
            A(state)
            for _ in range(int(k)):
                Q(state)
            return qp.probs(wires=[anc])

    _CIRCUIT_CACHE[key] = circuit
    return circuit


def circuit_sampler(flags: np.ndarray, use_catalyst: bool = True) -> Callable[[int, int], int]:
    """Returns sample_ones(k, shots) -> number of ancilla=|1> outcomes, measured on the circuit."""
    state = build_state(flags)
    n_wires = int(math.log2(state.shape[0]))

    def sample_ones(k: int, shots: int) -> int:
        circuit = get_grover_circuit(n_wires, shots, use_catalyst)
        state_arg = jnp.asarray(state) if use_catalyst else state
        p1 = float(np.asarray(circuit(state_arg, int(k)))[1])
        return int(round(p1 * shots))

    return sample_ones


def analytic_sampler(flags: np.ndarray, rng: Optional[np.random.Generator] = None) -> Callable[[int, int], int]:
    """Exact emulation of an ideal (noiseless) device: P(1 | k) = sin^2((2k+1) theta), theta = asin(sqrt(a)).

    Statistically identical to `circuit_sampler`, but O(1) per round. Useful for fast GAN training loops.
    """
    rng = rng if rng is not None else np.random.default_rng()
    a = float(np.clip(np.mean(np.clip(flags, 0.0, 1.0)), 0.0, 1.0))
    theta = math.asin(math.sqrt(a))

    def sample_ones(k: int, shots: int) -> int:
        p = min(max(math.sin((2 * k + 1) * theta) ** 2, 0.0), 1.0)
        return int(rng.binomial(shots, p))

    return sample_ones

## 3. Coherent QRAM oracle (LSH codes + Hamming radius)

The full state preparation $A$, with no classically computed flags:

$$A = O \cdot U_{\text{idx}}, \qquad U_{\text{idx}}|0\rangle = \tfrac{1}{\sqrt M}\sum_{i<M} |i\rangle,
\qquad O = C^\dagger \cdot \text{CMP}_{\le r} \cdot C$$

* $C$ computes the Hamming distance $h_i = \mathrm{popcount}(b_i \oplus q)$ into a counter register, one code bit at a time:
  1. **QRAM load**: $|i\rangle|0\rangle_{\text{data}} \to |i\rangle|b_{ij}\rangle$ (select-style QRAM, one multi-controlled $R_X(\pi b_{ij})$ per address),
  2. XOR the classical query bit $q_j$ into the data qubit,
  3. controlled increment: counter += data,
  4. undo step 2 and **unload** (QRAM$^\dagger$), returning the data qubit to $|0\rangle$.
* $\text{CMP}_{\le r}$ flips the flag qubit iff $h_i \le r$ (`qp.IntegerComparator`).
* $C^\dagger$ uncomputes the counter, leaving $\sum_i \tfrac{1}{\sqrt M}|i\rangle|0\rangle|0\rangle|[h_i\le r]\rangle$.

**Runtime-parameter QRAM:** the memory contents enter only as rotation angles $\pi b_{ij}$, the query only as $\pi q_j$, and the
occupied slots only through the index-superposition amplitudes. The circuit *structure* depends only on
$(\lceil\log_2 M\rceil, L, r)$, so Catalyst compiles once. Memory contents, the query, and even $M$ (up to the next power of two)
can then change without recompiling. $R_X(\pi) = -iX$ adds a branch-dependent phase $(-i)^{b_{ij}}$ on load, which the
matching unload ($R_X(-\pi b_{ij})$) cancels exactly.

Wires: `idx` ($\lceil\log_2 M\rceil$), `data` (1), `cnt` ($\lceil\log_2(L+1)\rceil$), `flag` (1). For example, 64 keys × 16-bit codes uses 13 qubits.
Select-style QRAM costs $O(M)$ gates per lookup but only $O(\log M)$ qubits (bucket-brigade would give $O(\log M)$ depth at $O(M)$ qubits).

Implementation note: `qp.adjoint` is applied to zero-argument closures so that Catalyst only traces the array arguments.

In [4]:
@dataclass(frozen=True)
class QRAMLayout:
    """Register layout. The circuit structure depends only on (n, L, radius), never on memory contents or M."""
    n: int          # address qubits -> 2**n QRAM slots (M <= 2**n keys; unused slots get zero amplitude)
    L: int          # code length (bits)
    radius: int     # Hamming radius

    @classmethod
    def for_memory(cls, M: int, L: int, radius: int) -> "QRAMLayout":
        return cls(num_index_qubits(M), L, radius)

    @property
    def slots(self) -> int:
        return 2 ** self.n

    @property
    def c(self) -> int:
        return max(1, math.ceil(math.log2(self.L + 1)))

    @property
    def idx(self) -> List[int]:
        return list(range(self.n))

    @property
    def data(self) -> int:
        return self.n

    @property
    def cnt(self) -> List[int]:  # MSB first
        return list(range(self.n + 1, self.n + 1 + self.c))

    @property
    def flag(self) -> int:
        return self.n + 1 + self.c

    @property
    def n_wires(self) -> int:
        return self.n + self.c + 2


def _index_bits(i: int, n: int) -> List[int]:
    return [(i >> (n - 1 - t)) & 1 for t in range(n)]  # wire 0 = MSB


def qram_load(layout: QRAMLayout, key_angles, j: int):
    """Select-style QRAM: data ^= b_ij for every address i (runtime angles pi*b_ij)."""
    for i in range(layout.slots):
        qp.ctrl(qp.RX, control=layout.idx, control_values=_index_bits(i, layout.n))(
            key_angles[i, j], wires=layout.data)


def controlled_increment(layout: QRAMLayout):
    """cnt += 1 if data == 1 (ripple of multi-controlled X, highest bit first)."""
    cnt, c = layout.cnt, layout.c
    for p in reversed(range(c)):             # p = bit position counted from the LSB
        target = cnt[c - 1 - p]
        lower = [cnt[c - 1 - t] for t in range(p)]
        qp.MultiControlledX(wires=[layout.data] + lower + [target])


def hamming_compute(layout: QRAMLayout, key_angles, query_angles):
    """C: |i>|0>|0> -> |i>|0>|popcount(b_i XOR q)>."""
    for j in range(layout.L):
        qram_load(layout, key_angles, j)
        qp.RX(query_angles[j], wires=layout.data)        # XOR q_j (up to global phase)
        controlled_increment(layout)
        qp.RX(-query_angles[j], wires=layout.data)
        qp.adjoint(lambda: qram_load(layout, key_angles, j))()   # unload: data back to |0>


def hamming_oracle(layout: QRAMLayout, key_angles, query_angles):
    """O: flips the flag qubit iff Hamming(b_i, q) <= radius; work registers are uncomputed."""
    if layout.radius >= layout.L:
        qp.PauliX(wires=layout.flag)                     # every key matches
        return
    hamming_compute(layout, key_angles, query_angles)
    qp.IntegerComparator(layout.radius + 1, geq=False, wires=layout.cnt + [layout.flag])
    qp.adjoint(lambda: hamming_compute(layout, key_angles, query_angles))()


def qram_A(layout: QRAMLayout, idx_state, key_angles, query_angles):
    """A = O . U_idx. idx_state (2**n,) is uniform over the M occupied slots (runtime argument)."""
    qp.MottonenStatePreparation(idx_state, wires=layout.idx)
    hamming_oracle(layout, key_angles, query_angles)


def qram_args(key_codes: np.ndarray, query_code: np.ndarray, layout: QRAMLayout):
    """Memory contents -> runtime circuit arguments (padded to 2**n slots)."""
    M = key_codes.shape[0]
    idx_state = np.zeros(layout.slots)
    idx_state[:M] = 1 / math.sqrt(M)
    ka = np.zeros((layout.slots, layout.L))
    ka[:M] = math.pi * np.asarray(key_codes, dtype=np.float64)
    qa = math.pi * np.asarray(query_code, dtype=np.float64)
    return idx_state, ka, qa


_QRAM_CACHE = {}


def get_qram_grover_circuit(layout: QRAMLayout, shots: Optional[int], use_catalyst: bool = True):
    """circuit(idx_state, key_angles, query_angles, k) -> probs of the flag qubit after Q^k A |0>."""
    key = (layout, shots, use_catalyst)
    if key in _QRAM_CACHE:
        return _QRAM_CACHE[key]

    wires = list(range(layout.n_wires))
    dev = qp.device("lightning.qubit", wires=layout.n_wires)

    def Q(s, ka, qa):
        qp.PauliZ(wires=layout.flag)                     # S_chi
        qp.adjoint(lambda: qram_A(layout, s, ka, qa))()  # A^dagger
        qp.FlipSign(0, wires=wires)                      # S_0
        qram_A(layout, s, ka, qa)                        # A

    if use_catalyst:
        @qp.qnode(dev, shots=shots)
        def circuit(s, ka, qa, k):
            qram_A(layout, s, ka, qa)

            @catalyst.for_loop(0, k, 1)
            def apply_Q(i):
                Q(s, ka, qa)

            apply_Q()
            return qp.probs(wires=[layout.flag])

        circuit = catalyst.qjit(circuit)
    else:
        @qp.qnode(dev, shots=shots)
        def circuit(s, ka, qa, k):
            qram_A(layout, s, ka, qa)
            for _ in range(int(k)):
                Q(s, ka, qa)
            return qp.probs(wires=[layout.flag])

    _QRAM_CACHE[key] = circuit
    return circuit


def qram_sampler(key_codes: np.ndarray, query_code: np.ndarray, radius: int,
                 use_catalyst: bool = True) -> Callable[[int, int], int]:
    """key_codes (M, L) in {0,1}, query_code (L,) in {0,1}. Matching is computed *in the circuit*."""
    key_codes = np.asarray(key_codes)
    layout = QRAMLayout.for_memory(key_codes.shape[0], key_codes.shape[1], int(radius))
    args = qram_args(key_codes, query_code, layout)
    if use_catalyst:
        args = tuple(jnp.asarray(a) for a in args)

    def sample_ones(k: int, shots: int) -> int:
        circuit = get_qram_grover_circuit(layout, shots, use_catalyst)
        p1 = float(np.asarray(circuit(*args, int(k)))[1])
        return int(round(p1 * shots))

    return sample_ones


def qram_oracle_check(key_codes: np.ndarray, query_code: np.ndarray, radius: int):
    """Exact statevector check of A: returns (count encoded in P(flag=1), P(work registers == 0))."""
    key_codes = np.asarray(key_codes)
    layout = QRAMLayout.for_memory(key_codes.shape[0], key_codes.shape[1], int(radius))
    dev = qp.device("lightning.qubit", wires=layout.n_wires)

    @qp.qnode(dev)
    def circuit(s, ka, qa):
        qram_A(layout, s, ka, qa)
        return qp.probs(wires=[layout.flag]), qp.probs(wires=[layout.data] + layout.cnt)

    p_flag, p_work = circuit(*qram_args(key_codes, query_code, layout))
    return float(p_flag[1]) * key_codes.shape[0], float(p_work[0])


def qram_resources(M: int, L: int, radius: int) -> dict:
    """Qubits and top-level gate counts for one application of A (each Grover iterate Q uses A and A^dagger)."""
    layout = QRAMLayout.for_memory(M, L, radius)
    s, ka, qa = qram_args(np.zeros((M, L)), np.zeros(L), layout)
    tape = qp.tape.make_qscript(qram_A)(layout, s, ka, qa)
    counts = {}
    for op in tape.operations:
        counts[op.name] = counts.get(op.name, 0) + 1
    return {"qubits": layout.n_wires, "qram_lookups_per_A": 4 * L if radius < L else 0,
            "ops_per_A": len(tape.operations), "by_type": counts}

## 4. IQAE (Grinko et al., Chernoff-Hoeffding intervals)

Fixes compared with the original implementation:
* **Shot pooling**: when `find_next_k` keeps the same `k`, the shots from those rounds are pooled, as the
  algorithm requires. The original code discarded them, which could stall forever and broke the confidence guarantee.
* Guard against a zero-width interval (division by zero in `K_max`).
* Handling for non-overlapping (statistically inconsistent) intervals, plus a `max_rounds` safety net.
* `k` stays an integer, and the measurement function is decoupled from the algorithm (QRAM circuit, baseline circuit, or analytic).
* Stops when either the $\theta$ interval or the $a$ interval reaches width $2\varepsilon$
  ($|\Delta a| \le |\Delta\theta|$, so the round bound $T$ and the $1-\alpha$ guarantee still hold).

In [5]:
def find_next_k(k: int, theta_l: float, theta_u: float, upper_half: bool, r: int = 2):
    """Largest K = 4k'+2 (>= r*K_i) such that K*[theta_l, theta_u] lies in a single half-plane mod 2pi."""
    K_i = 4 * k + 2
    width = theta_u - theta_l
    if width <= 0:
        return k, upper_half
    K_max = math.floor(math.pi / width)
    K = K_max - (K_max - 2) % 4
    two_pi = 2 * math.pi
    while K >= r * K_i:
        lo = (K * theta_l) % two_pi
        hi = (K * theta_u) % two_pi
        if lo <= math.pi and hi <= math.pi:
            return (K - 2) // 4, True
        if lo >= math.pi and hi >= math.pi:
            return (K - 2) // 4, False
        K -= 4
    return k, upper_half


@dataclass
class IQAEResult:
    a_lower: float
    a_upper: float
    rounds: int
    grover_calls: int          # total applications of Q across all shots
    ks: List[int] = field(default_factory=list)
    inconsistent_rounds: int = 0

    @property
    def a(self) -> float:
        return 0.5 * (self.a_lower + self.a_upper)


def iqae(sample_ones: Callable[[int, int], int], eps: float, alpha: float, shots: int,
         max_rounds: Optional[int] = None) -> IQAEResult:
    """Estimate a in [0,1] to within +-eps with confidence 1 - alpha."""
    if not (0 < eps < 0.5):
        raise ValueError("eps must be in (0, 0.5)")
    if not (0 < alpha < 1):
        raise ValueError("alpha must be in (0, 1)")

    T = max(1, math.ceil(math.log2(math.pi / (8 * eps))))
    log_term = math.log(2 * T / alpha)
    max_rounds = max_rounds if max_rounds is not None else 100 * T

    theta_l, theta_u = 0.0, math.pi / 2
    k, upper_half = 0, True
    pooled_ones = pooled_shots = 0
    rounds = grover_calls = inconsistent = 0
    ks = []

    def a_width():
        return math.sin(theta_u) ** 2 - math.sin(theta_l) ** 2

    while theta_u - theta_l > 2 * eps and a_width() > 2 * eps:
        if rounds >= max_rounds:
            warnings.warn(f"IQAE hit max_rounds={max_rounds}; returning current interval.")
            break

        k_new, upper_half = find_next_k(k, theta_l, theta_u, upper_half)
        if k_new != k:
            pooled_ones = pooled_shots = 0
        k = k_new

        pooled_ones += sample_ones(k, shots)
        pooled_shots += shots
        rounds += 1
        grover_calls += k * shots
        ks.append(k)

        p_hat = pooled_ones / pooled_shots
        eps_p = math.sqrt(log_term / (2 * pooled_shots))
        p_min = min(max(p_hat - eps_p, 0.0), 1.0)
        p_max = min(max(p_hat + eps_p, 0.0), 1.0)

        if upper_half:
            est_l = 2 * math.asin(math.sqrt(p_min))
            est_u = 2 * math.asin(math.sqrt(p_max))
        else:
            est_l = 2 * math.pi - 2 * math.asin(math.sqrt(p_max))
            est_u = 2 * math.pi - 2 * math.asin(math.sqrt(p_min))

        K = 4 * k + 2
        two_pi = 2 * math.pi
        new_l = (two_pi * math.floor(K * theta_l / two_pi) + est_l) / K
        new_u = (two_pi * math.floor(K * theta_u / two_pi) + est_u) / K

        if new_l > theta_u or new_u < theta_l:
            # Statistically inconsistent with the previous interval (probability <= alpha overall).
            # Trust the newest, highest-K measurement.
            inconsistent += 1
            theta_l, theta_u = new_l, new_u
        else:
            theta_l, theta_u = max(theta_l, new_l), min(theta_u, new_u)

    theta_l = min(max(theta_l, 0.0), math.pi / 2)
    theta_u = min(max(theta_u, theta_l), math.pi / 2)
    return IQAEResult(math.sin(theta_l) ** 2, math.sin(theta_u) ** 2, rounds, grover_calls, ks, inconsistent)

## 5. M-GAN hook: `QAEMemoryCounter`

```python
matcher = LSHHamming(dim=d, n_bits=16, radius_bits=3)
counter = QAEMemoryCounter(matcher, backend="qram")
out = counter(query_embeddings, memory_keys)  # query (d,) or (B, d); keys (M, d)
out.count          # (B,) differentiable neighbour-count estimate (forward = QAE, backward = surrogate)
out.count_lower    # (B,) lower confidence bound (detached)
out.count_upper    # (B,) upper confidence bound (detached)
```

* `count_eps`: target half-width of the count interval, in **number of keys** (0.5 resolves integer counts).
* `backend`: `"qram"` (coherent QRAM + Hamming oracle), `"stateprep"` (classical-flag baseline), or `"analytic"`
  (ideal-device statistics, fastest).
* The QRAM circuit is compiled once per `(ceil(log2 M), n_bits, radius_bits, shots)`. Updating memory contents, or
  growing M up to the next power of two, does not trigger a recompile.

In [6]:
class QAECountOutput(NamedTuple):
    count: torch.Tensor
    count_lower: torch.Tensor
    count_upper: torch.Tensor
    grover_calls: torch.Tensor
    results: list


class QAEMemoryCounter(torch.nn.Module):
    BACKENDS = ("qram", "stateprep", "analytic")

    def __init__(self, matcher, count_eps: float = 0.5, alpha: float = 0.05, shots: int = 1024,
                 backend: str = "qram", use_catalyst: bool = True, max_qubits: int = 20,
                 surrogate_grad: bool = True, seed: Optional[int] = None):
        super().__init__()
        if backend not in self.BACKENDS:
            raise ValueError(f"backend must be one of {self.BACKENDS}")
        if backend == "qram" and not (hasattr(matcher, "codes") and hasattr(matcher, "radius_bits")):
            raise ValueError("backend='qram' needs a matcher with codes() and radius_bits (e.g. LSHHamming)")
        self.matcher = matcher
        self.count_eps = count_eps
        self.alpha = alpha
        self.shots = shots
        self.backend = backend
        self.use_catalyst = use_catalyst
        self.max_qubits = max_qubits
        self.surrogate_grad = surrogate_grad
        self.rng = np.random.default_rng(seed)

    def _check_qubits(self, n_wires: int, M: int):
        if n_wires > self.max_qubits:
            raise ValueError(f"{M} keys need {n_wires} qubits (> max_qubits={self.max_qubits}); "
                             "use backend='analytic' or raise max_qubits.")

    def _samplers(self, q: torch.Tensor, keys: torch.Tensor) -> list:
        M, B = keys.shape[0], q.shape[0]
        with torch.no_grad():
            if self.backend == "qram":
                key_codes = self.matcher.codes(keys).cpu().numpy()   # what is stored in QRAM
                q_codes = self.matcher.codes(q).cpu().numpy()
                layout = QRAMLayout.for_memory(M, key_codes.shape[1], self.matcher.radius_bits)
                self._check_qubits(layout.n_wires, M)
                return [qram_sampler(key_codes, q_codes[b], self.matcher.radius_bits, self.use_catalyst)
                        for b in range(B)]
            flags = self.matcher.hard(q, keys).detach().cpu().double().numpy()  # (B, M)
            if self.backend == "analytic":
                return [analytic_sampler(flags[b], self.rng) for b in range(B)]
            self._check_qubits(num_index_qubits(M) + 1, M)
            return [circuit_sampler(flags[b], self.use_catalyst) for b in range(B)]

    def forward(self, query: torch.Tensor, keys: torch.Tensor) -> QAECountOutput:
        squeeze = query.dim() == 1
        q = query.unsqueeze(0) if squeeze else query
        M, B = keys.shape[0], q.shape[0]
        if M == 0:
            z = torch.zeros(B, dtype=q.dtype, device=q.device)
            z = z[0] if squeeze else z
            return QAECountOutput(z, z, z, torch.zeros_like(z, dtype=torch.long), [])

        eps_a = min(self.count_eps / M, 0.49)
        results = [iqae(s, eps_a, self.alpha, self.shots) for s in self._samplers(q, keys)]
        to_t = lambda xs, dt=q.dtype: torch.tensor(xs, dtype=dt, device=q.device)
        qae_count = to_t([r.a * M for r in results])
        lower = to_t([r.a_lower * M for r in results])
        upper = to_t([r.a_upper * M for r in results])
        calls = to_t([r.grover_calls for r in results], torch.long)

        if self.surrogate_grad and (q.requires_grad or keys.requires_grad):
            soft = self.matcher.soft(q, keys).sum(dim=-1)
            count = soft + (qae_count - soft).detach()   # value = QAE, gradient = d(soft count)
        else:
            count = qae_count

        if squeeze:
            count, lower, upper, calls = count[0], lower[0], upper[0], calls[0]
        return QAECountOutput(count, lower, upper, calls, results)

## 6. Demo: real vs fake queries against a memory of keys (QRAM backend)

In [7]:
torch.manual_seed(0)
d, M = 16, 60
centers = torch.nn.functional.normalize(torch.randn(3, d), dim=-1)
keys = torch.nn.functional.normalize(centers.repeat_interleave(M // 3, 0) + 0.25 * torch.randn(M, d), dim=-1)

real_q = torch.nn.functional.normalize(centers[[0, 1]] + 0.25 * torch.randn(2, d), dim=-1)  # near stored keys
fake_q = torch.nn.functional.normalize(torch.randn(2, d), dim=-1)                          # unrelated
queries = torch.cat([real_q, fake_q]).requires_grad_(True)
labels = ["real", "real", "fake", "fake"]

matcher = LSHHamming(dim=d, n_bits=16, radius_bits=3, seed=0)
exact = matcher.hard(queries, keys).sum(-1)   # classical reference only (not used by the QRAM backend)
layout = QRAMLayout.for_memory(M, matcher.n_bits, matcher.radius_bits)

counter = QAEMemoryCounter(matcher, count_eps=0.5, alpha=0.05, shots=1024, backend="qram")
t0 = time.time()
out = counter(queries, keys)
t1 = time.time()

print(f"QRAM backend: {layout.n_wires} qubits ({layout.n} address, 1 data, {layout.c} counter, 1 flag), "
      f"{t1 - t0:.1f}s incl. Catalyst compile")
for i, label in enumerate(labels):
    print(f"  {label}: exact={exact[i].item():4.0f}  QAE={out.count[i].item():6.2f}  "
          f"CI=[{out.count_lower[i].item():6.2f}, {out.count_upper[i].item():6.2f}]  "
          f"rounds={out.results[i].rounds:2d}  grover_calls={out.grover_calls[i].item()}")

# Gradient flows to the input embedding through the surrogate
out.count.sum().backward()
print("grad norm per query:", [round(g, 4) for g in queries.grad.norm(dim=-1).tolist()])

QRAM backend: 13 qubits (6 address, 1 data, 5 counter, 1 flag), 43.5s incl. Catalyst compile
  real: exact=  17  QAE= 16.92  CI=[ 16.52,  17.32]  rounds= 2  grover_calls=3072
  real: exact=   8  QAE=  7.96  CI=[  7.61,   8.30]  rounds= 2  grover_calls=3072
  fake: exact=   0  QAE=  0.06  CI=[  0.00,   0.13]  rounds= 2  grover_calls=2048
  fake: exact=   0  QAE=  0.06  CI=[  0.00,   0.13]  rounds= 2  grover_calls=2048
grad norm per query: [17.0969, 59.9696, 1.8686, 0.0256]


In [8]:
# Cached: new query values and changed memory contents reuse the compiled circuit
t0 = time.time()
out2 = counter(queries.detach(), keys)
print(f"Cached QRAM call: {time.time() - t0:.2f}s, counts = {[round(c, 2) for c in out2.count.tolist()]}")

# Memory update: replace 10 keys and drop to M=50 (same 6 address qubits -> no recompile)
new_keys = keys.clone()[:50]
new_keys[:10] = torch.nn.functional.normalize(fake_q[0] + 0.2 * torch.randn(10, d), dim=-1)
t0 = time.time()
out_upd = counter(queries.detach(), new_keys)
print(f"Updated memory (M=50): {time.time() - t0:.2f}s, exact = {matcher.hard(queries, new_keys).sum(-1).tolist()}, "
      f"QAE = {[round(c, 2) for c in out_upd.count.tolist()]}")

# Other backends, same interface
for backend in ("stateprep", "analytic"):
    other = QAEMemoryCounter(matcher, count_eps=0.5, alpha=0.05, shots=1024, backend=backend, seed=1)
    t0 = time.time()
    o = other(queries.detach(), keys)
    print(f"{backend:>9} backend: {time.time() - t0:.3f}s, counts = {[round(c, 2) for c in o.count.tolist()]}")

Cached QRAM call: 0.17s, counts = [16.81, 8.0, 0.06, 0.06]
Updated memory (M=50): 0.10s, exact = [9.0, 8.0, 6.0, 0.0], QAE = [9.12, 7.93, 5.91, 0.05]


stateprep backend: 1.626s, counts = [17.06, 8.0, 0.06, 0.06]
 analytic backend: 0.001s, counts = [17.18, 7.97, 0.06, 0.06]


### QRAM resources
Qubit count and top-level operation count for one application of $A$. Each Grover iterate $Q$ applies $A$ and
$A^\dagger$. Each QRAM lookup is $2^{\lceil\log_2 M\rceil}$ multi-controlled $R_X$ gates (select-style).

In [9]:
for M_res in (64, 256, 1024):
    res = qram_resources(M_res, 16, 3)
    print(f"M={M_res:5d}, L=16: qubits={res['qubits']:2d}, QRAM lookups/A={res['qram_lookups_per_A']}, "
          f"ops/A={res['ops_per_A']}")

M=   64, L=16: qubits=13, QRAM lookups/A=64, ops/A=4322


M=  256, L=16: qubits=15, QRAM lookups/A=64, ops/A=16610


M= 1024, L=16: qubits=17, QRAM lookups/A=64, ops/A=65762


## 7. Validation
1. **QRAM oracle exactness:** an exact (shot-free) statevector check that $A$ encodes exactly the classical Hamming-radius
   count, and that the data and counter registers are fully uncomputed.
2. **Coverage:** over many random amplitudes, the $1-\alpha$ interval should contain the true value at least $1-\alpha$
   of the time (analytic sampler, statistically identical to an ideal circuit).
3. **End-to-end:** QRAM-backend IQAE and baseline-circuit IQAE against exact counts.

In [10]:
rng = np.random.default_rng(123)

print("1) QRAM oracle exactness")
for (M_t, L_t, r_t) in [(5, 4, 1), (8, 6, 2), (13, 8, 3), (16, 8, 0), (16, 8, 8)]:
    kc = rng.integers(0, 2, (M_t, L_t)); qc = rng.integers(0, 2, L_t)
    exact_t = int(((kc != qc).sum(1) <= r_t).sum())
    count_t, clean_t = qram_oracle_check(kc, qc, r_t)
    print(f"   M={M_t:2d} L={L_t} r={r_t}: exact={exact_t:2d}  circuit={count_t:.6f}  work registers clean={clean_t:.6f}")

print("2) IQAE coverage")
eps, alpha, shots = 1e-3, 0.05, 256
trials, hits, widths, calls = 300, 0, [], []
for _ in range(trials):
    a_true = rng.uniform(0, 1)
    r = iqae(analytic_sampler(np.array([a_true]), rng), eps, alpha, shots)
    hits += r.a_lower - 1e-12 <= a_true <= r.a_upper + 1e-12
    widths.append(r.a_upper - r.a_lower)
    calls.append(r.grover_calls)
print(f"   coverage = {hits / trials:.3f} (target >= {1 - alpha}),  max width = {max(widths):.2e} "
      f"(target <= {2 * eps}),  mean Grover calls = {np.mean(calls):.0f}")
for a_true in (0.0, 1.0):
    r = iqae(analytic_sampler(np.array([a_true]), rng), eps, alpha, shots)
    print(f"   a={a_true}: interval [{r.a_lower:.5f}, {r.a_upper:.5f}], rounds={r.rounds}")

print("3) End-to-end vs exact")
for M_t in (9, 12, 16):   # all share one compiled QRAM circuit (4 address qubits)
    kc = rng.integers(0, 2, (M_t, 8)); qc = kc[0] ^ (rng.uniform(size=8) < 0.2)
    exact_t = int(((kc != qc).sum(1) <= 2).sum())
    r = iqae(qram_sampler(kc, qc, 2), 0.5 / M_t, alpha, 1024)
    lo, hi = r.a_lower * M_t, r.a_upper * M_t
    print(f"   qram      M={M_t:2d}: exact={exact_t:2d}  CI=[{lo:6.2f}, {hi:6.2f}]  contains={lo - 1e-9 <= exact_t <= hi + 1e-9}")
for M_t in (5, 16, 37):
    flags = (rng.uniform(size=M_t) < 0.3).astype(float)
    r = iqae(circuit_sampler(flags), 0.5 / M_t, alpha, 1024)
    lo, hi = r.a_lower * M_t, r.a_upper * M_t
    print(f"   stateprep M={M_t:2d}: exact={flags.sum():2.0f}  CI=[{lo:6.2f}, {hi:6.2f}]  "
          f"contains={lo - 1e-9 <= flags.sum() <= hi + 1e-9}")

1) QRAM oracle exactness
   M= 5 L=4 r=1: exact= 2  circuit=2.000000  work registers clean=1.000000
   M= 8 L=6 r=2: exact= 4  circuit=4.000000  work registers clean=1.000000
   M=13 L=8 r=3: exact= 3  circuit=3.000000  work registers clean=1.000000
   M=16 L=8 r=0: exact= 0  circuit=0.000000  work registers clean=1.000000
   M=16 L=8 r=8: exact=16  circuit=16.000000  work registers clean=1.000000
2) IQAE coverage
   coverage = 0.997 (target >= 0.95),  max width = 1.98e-03 (target <= 0.002),  mean Grover calls = 36652
   a=0.0: interval [0.00000, 0.00066], rounds=3
   a=1.0: interval [0.99908, 1.00000], rounds=3
3) End-to-end vs exact


   qram      M= 9: exact= 3  CI=[  2.57,   3.44]  contains=True
   qram      M=12: exact= 2  CI=[  1.94,   2.15]  contains=True
   qram      M=16: exact= 5  CI=[  4.89,   5.07]  contains=True


   stateprep M= 5: exact= 1  CI=[  0.73,   1.19]  contains=True


   stateprep M=16: exact= 7  CI=[  6.77,   7.16]  contains=True
   stateprep M=37: exact=14  CI=[ 13.76,  14.12]  contains=True
